This was my first working notebook for the YOLO side of the project,
before I split things out into the separate `modelfall.ipynb` and the
scripts folder. It's a mix of things: training the first "optimized" run,
checking predictions visually, a k-fold stability check, and two attempts
at wiring the model up to real hardware over MQTT (an ESP32-CAM stream in
one version, a folder of test images in the other) for the fall-detection
alert system. The cells are independent experiments, not a pipeline meant
to run top to bottom.

Just confirms the right CUDA-enabled PyTorch is actually installed before training anything.

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)           # should report "12.1"
print(torch.cuda.is_available())    # should be True
print(torch.cuda.device_count())    # 1 (or more)


The first real training run on this dataset version, using the settings I ended up sticking with (cosine LR, warmup, SGD).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="new_dataset/data.yaml",
    epochs=100,
    batch=8,
    imgsz=640,
    device="cuda",
    workers=4,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.1, 
    momentum=0.937, 
    weight_decay=0.0005,
    cos_lr=True,
    patience=10, 
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    project="runs/train",
    name="Optimized_Fall_Model",
    exist_ok=True
)


Runs the trained model over the validation images and saves the annotated detections so I can look through them by eye.

In [ ]:
from ultralytics import YOLO
import os
import cv2

# Panggil model
model = YOLO("runs/train/Optimized_Fall_Model/weights/best.pt")  # atau sesuaikan path model Anda

# Panggil direktori input dan output
image_dir = "new_dataset/images/val"  # Folder berisi gambar validasi
output_dir = "new_dataset/visuals_new"  # Folder hasil deteksi
os.makedirs(output_dir, exist_ok=True)

# Loop semua gambar
for filename in os.listdir(image_dir):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.join(image_dir, filename)

        # Lakukan deteksi
        results = model(image_path)

        # Simpan hasil dengan anotasi bounding box
        for r in results:
            annotated_frame = r.plot()
            output_path = os.path.join(output_dir, filename)
            cv2.imwrite(output_path, annotated_frame)

print("Deteksi selesai dan hasil disimpan di:", output_dir)


Same k-fold stability check as in the other notebooks, just against this earlier dataset version and model.

In [ ]:
import os
import shutil
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

k = 5
DATASET_BASE = "dataset_paper_new"
ORIG_VAL_IMG = f"{DATASET_BASE}/images/train"
ORIG_VAL_LBL = f"{DATASET_BASE}/labels/train"
TEMP_DIR = "temp_kfold_yolo"
MODEL_PATH = "runs/train/YOLO_noD_reverse_lr.0.001/weights/best.pt"

Path(TEMP_DIR).mkdir(exist_ok=True)

all_files = sorted([f for f in os.listdir(ORIG_VAL_IMG) if f.endswith((".jpg", ".png"))])
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

model = YOLO(MODEL_PATH)

metrics_list = []

for fold, (_, val_idx) in enumerate(kfold.split(all_files), 1):
    print(f"\n🔁 Fold {fold}")

    fold_base = Path(TEMP_DIR) / f"fold{fold}"
    img_dir = fold_base / "images"
    lbl_dir = fold_base / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for i in val_idx:
        fname = all_files[i]
        stem = os.path.splitext(fname)[0]
        shutil.copy(os.path.join(ORIG_VAL_IMG, fname), img_dir / fname)
        shutil.copy(os.path.join(ORIG_VAL_LBL, f"{stem}.txt"), lbl_dir / f"{stem}.txt")

    temp_yaml_path = fold_base / "data_nano_rev.yaml"
    with open(f"{DATASET_BASE}/data_nano_rev.yaml", 'r') as f:
        data_yaml = yaml.safe_load(f)

    data_yaml["val"] = os.path.abspath(img_dir)

    with open(temp_yaml_path, 'w') as f:
        yaml.dump(data_yaml, f)

    results = model.val(
        data=str(temp_yaml_path),
        imgsz=640,
        batch=8,
        iou=0.5,
        conf=0.001,
        verbose=False
    )

    precision = results.box.mp if hasattr(results, "box") else 0.0
    recall = results.box.mr if hasattr(results, "box") else 0.0

    # Compute F1 Score
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0

    # Compute Accuracy (approximate)
    if precision + recall + abs(precision - recall) > 0:
        accuracy = (2 * precision * recall) / (precision + recall + abs(precision - recall))
    else:
        accuracy = 0.0

    # Collect metrics
    metrics_list.append({
        "precision": precision,
        "recall": recall,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "mAP50": results.box.map50 if hasattr(results, "box") else 0.0,
        "mAP50-95": results.box.map if hasattr(results, "box") else 0.0
    })
# Cleanup
shutil.rmtree(TEMP_DIR)

# Create DataFrame
df = pd.DataFrame(metrics_list, index=[f"Fold {i}" for i in range(1, k + 1)])
summary = df.agg(['mean', 'std']).rename(index={'mean': 'Mean', 'std': 'Std'})

print("\n=== Per-Fold Metrics ===")
print(df.to_string(float_format="%.4f"))

print("\n=== Cross-Validation Summary ===")
print(summary.to_string(float_format="%.4f"))


First attempt at connecting the model to a live camera feed: pulls frames
from an ESP32-CAM stream, runs detection, and publishes a fall/no-fall
state to an MQTT broker so the rest of the alert system can react to it.

In [ ]:
import cv2
import time
from ultralytics import YOLO
import paho.mqtt.client as mqtt

STREAM_URL  = "http://192.168.1.100:81/stream"
MODEL_PATH  = "best.pt"
BROKER      = "192.168.88.25"
PORT        = 1883
TOPIC       = "fall/state"
USERNAME    = "username_mqtt"
PASSWORD    = "password_mqtt"
CONF_THRESH = 0.5
FALL_CLASS  = 1
INTERVAL    = 2.0

client = mqtt.Client()
client.username_pw_set(USERNAME,PASSWORD)
client.connect(BROKER, PORT)
client.loop_start()

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(STREAM_URL)

last_sent = 0

while True:
    ret, frame = cap.read()
    if not ret:
        time.sleep(0.1)
        continue

    results = model.predict(frame, conf=CONF_THRESH, verbose=False)[0]

    for box in results.boxes:
        if int(box.cls) == FALL_CLASS and box.conf >= CONF_THRESH:
            now = time.time()
            if now - last_sent > INTERVAL:
                client.publish(TOPIC, "fall")
                last_sent = now
            break

    time.sleep(0.01)


Quick model summary printout, mostly just to check the architecture and parameter count.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

print(model.info())

Second version of the hardware integration, this time reading from a
folder of test images on a delay instead of a live stream, which was
easier for testing the MQTT publishing side without needing the camera
running.

In [ ]:
import os
import time
from datetime import datetime
from ultralytics import YOLO
import paho.mqtt.client as mqtt

# ─── CONFIG ──────────────────────────────────────
IMAGE_DIR = "test_fall"
MODEL_PATH = "best.pt"
BROKER = "192.168.88.25"
PORT = 1883
TOPIC = "fall/state"
USERNAME = "username_mqtt"
PASSWORD = "password_mqtt"
CONF_THRESH = 0.5
FALL_CLASS = 1
DELAY = 15  # seconds between images
# ─────────────────────────────────────────────────

client = mqtt.Client()
client.username_pw_set(USERNAME, PASSWORD)
client.connect(BROKER, PORT)
client.loop_start()

model = YOLO(MODEL_PATH)

for filename in os.listdir(IMAGE_DIR):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(IMAGE_DIR, filename)

    start_time = time.time()
    start_str = datetime.now().strftime("%H:%M:%S.%f")[:-3]

    results = model(image_path)[0]

    status = "no_detection"

    if results.boxes is not None and len(results.boxes) > 0:
        status = "no_fall"  # asumsi ada deteksi tapi belum ada "fall"
        for box in results.boxes:
            if int(box.cls) == FALL_CLASS and box.conf >= CONF_THRESH:
                status = "fall"
                break

    client.publish(TOPIC, status)

    end_time = time.time()
    end_str = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    duration = end_time - start_time

    print(f"[INFO] Mulai deteksi   : {start_str}")
    print(f"[INFO] Dikirim ke MQTT : {end_str}")
    print(f"[INFO] Durasi proses   : {duration:.3f} detik")
    print(f"[INFO] Status '{status}' untuk file '{filename}'\n")

    time.sleep(DELAY)

client.loop_stop()
client.disconnect()
print("Deteksi selesai.")
